# Analysis Notebook

This notebook contains the computational analysis and findings presented in the paper. It begins with the full population of 14,236 rank-49 AlphaTensor decompositions and reconstructs the results from the underlying data.

Specifically, we:

### 1. Recover Rules 0–3 from the Population

* Recover the four highest-overlap algorithm pairs.
* Identify their shared terms and residual terms.
* Construct the corresponding residual tensor identities.

### 2. Structural Screening of Rules 0–3

For each rule:

* Confirm that the two residual decompositions are exactly equal.
* Confirm that there are no smaller proper subidentities.
* Identify the number of nonzero tensor entries.
* Compute the multilinear rank.
* Compute the factor-span dimensions.
* Check for proportional or shared factors.
* Determine the residual support.
* Check for simple mode symmetries.

### 3. Geometric Screening of Rules 0–3

For each residual:

* Reduce the residual to its intrinsic mode spaces.
* Compute the CP Jacobian.
* Identify and remove the $2r$ ordinary per-term scaling directions.
* Determine the number of non-gauge local deformation directions.
* Compute the infinitesimal residual stabilizer.
* Remove the scalar stabilizer directions.
* Determine the number of nontrivial stabilizer directions.
* Compare the resulting local deformation and stabilizer dimensions across Rules 0–3.

### 4. Investigate Rules 1–3

Test whether the two residual decompositions associated with Rules 1–3 can be explained by simple discrete transformations:

* Mode permutations.
* Cyclic rotations.
* Transpositions.
* Term permutations.
* Termwise rescalings.

Use these results, together with the structural and geometric screening, to explain why the remainder of the notebook focuses on Rule 0.

### 5. Recover and Analyze Rule 0's $5\leftrightarrow5$ Identity

* Display the ten rank-one terms forming the two five-term realizations.
* Verify the exact $5\leftrightarrow5$ residual identity.
* Establish the nine-entry support of the residual tensor $R$.
* Construct the intrinsic $4\times4\times3$ representation of $R$.

### 6. Analyze the 44-Term Core

* Construct the 44-term common core.
* Show that the core computes exactly 13 of the 16 entries of $AB$.
* Derive the exact errors in:

  * $C_{33}$,
  * $C_{34}$,
  * $C_{43}$.
* Analyze the partial cores obtained from 45, 46, 47, and 48 terms and report the corresponding output/error structure.

### 7. Population-Wide Rule-0 Rewrite

* Search all 14,236 algorithms for either five-term realization of Rule 0.
* Identify every valid occurrence of the rewrite.
* Swap the two five-term realizations.
* Verify every generated candidate exactly against $T_{444}$.
* Record the source algorithm, rewrite direction, matched terms, and verification result.

### 8. Establish the $K$-Invariant Results

* Reproduce DeepMind's published $K$-invariant results on the released population.
* Compute the $K$-invariant for the Rule-0-generated candidates.
* Identify distinct candidate $K$-signatures.
* Determine which signatures occur in the released population.
* Determine which candidate signatures are absent from the released $K$-applicable population.

### 9. CP Geometry of Rule 0

* Perform the detailed CP Jacobian calculation for the intrinsic $4\times4\times3$ residual.
* Determine the exact Jacobian rank and nullity.
* Construct the ordinary per-term scaling directions explicitly.
* Verify that they lie in the Jacobian kernel.
* Isolate the unique non-gauge local deformation direction.

### 10. Residual Stabilizer

* Solve the infinitesimal stabilizer equations for $R$.
* Obtain the three-dimensional Lie stabilizer.
* Identify the two scalar directions.
* Isolate the unique nontrivial generator.
* Compare this direction with the unique non-gauge direction obtained from the CP Jacobian.

### 11. Derive the $t$-Family

* Integrate the unique nontrivial stabilizer generator.
* Prove symbolically that the resulting transformation fixes $R$.
* Generate the five parameterized rank-one terms.
* Verify the parameter values corresponding to Algorithms 5832 and 7414.
* Show that the two discrete Rule-0 realizations lie on the same continuous family.
* Transplant the $t$-family to the population algorithms containing Rule 0.

### 12. Rule Out a Global $PQR$ Explanation

* Compute the connected $PQR$ stabilizer of the 44-term core.
* Determine its exact rank and nullity.
* Show that the three remaining dimensions consist only of scalar gauge transformations.
* Conclude that there is no nontrivial connected global $PQR$ transformation fixing the 44-term core that explains the $t$-family.

### 13. Numerical Stability Analysis

* Run a controlled floating-point experiment over $t$.
* Measure the coefficient dynamic range as $t$ varies.
* Measure the resulting floating-point error.
* Compare numerical behavior near the original parameter values with behavior at increasingly extreme values of $t$.
* Relate coefficient growth to the observed degradation in numerical stability.


In [37]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path

## 1. Recover Rules 0–3 from the Population

In [38]:
# Recover the four highest-overlap algorithm pairs
data_directory = Path('../data_processed')
top_pairs = pd.read_csv(data_directory/'top_pairs.csv')
rule_pairs = top_pairs.iloc[:4].copy()
print(rule_pairs)

canonical_data = np.load(data_directory/'canonical_population.npz')
print(canonical_data.files) # should be ['canonical_population']
canonical_population = canonical_data['canonical_population']
print(canonical_population.shape) # should be (14236, 49, 3, 16)

   algorithm_i  algorithm_j  shared_terms
0         5832         7414            44
1         9105        10222            43
2          913         1841            42
3         1618         2156            42
['canonical_population']
(14236, 49, 3, 16)


In [39]:
# Identify their shared terms and residual terms
def identify_residual_terms(alg_i, alg_j):
    alg_i_array = canonical_population[alg_i]
    alg_j_array = canonical_population[alg_j]

    residual_i = []
    residual_j = []

    for index, term_i in enumerate(alg_i_array):

        match_found = False

        for term_j in alg_j_array:
            if np.array_equal(term_i, term_j):
                match_found = True
                break

        if not match_found:
            residual_i.append(index)

    for index, term_j in enumerate(alg_j_array):

        match_found = False

        for term_i in alg_i_array:
            if np.array_equal(term_j, term_i):
                match_found = True
                break

        if not match_found:
            residual_j.append(index)

    print(residual_i, residual_j)
    return residual_i, residual_j

rule_residual_indices = []

for algorithm_index in range(4):
    alg_i = rule_pairs.iloc[algorithm_index]['algorithm_i']
    alg_j = rule_pairs.iloc[algorithm_index]['algorithm_j']

    residual_i, residual_j = identify_residual_terms(alg_i, alg_j)

    rule_residual_indices.append({
        'algorithm_i': alg_i,
        'algorithm_j': alg_j,
        'residual_i': residual_i,
        'residual_j': residual_j
    })

[0, 1, 2, 3, 43] [0, 1, 2, 5, 6]
[10, 13, 14, 15, 19, 20] [12, 13, 14, 15, 16, 47]
[37, 39, 40, 41, 44, 46, 48] [38, 39, 40, 41, 43, 44, 45]
[32, 33, 34, 35, 36, 37, 44] [30, 31, 32, 33, 35, 38, 42]


In [40]:
# Construct the corresponding residual tensor identities
def tensor_from_terms(terms):
    tensor = np.zeros((16, 16, 16), dtype=int)

    for term in terms:
        u = term[0]
        v = term[1]
        w = term[2]

        tensor += np.einsum('i,j,k->ijk', u, v, w)

    return tensor

for rule in rule_residual_indices:
    alg_i = rule['algorithm_i']
    alg_j = rule['algorithm_j']
    residual_i_indices = rule['residual_i']
    residual_j_indices = rule['residual_j']
    
    alg_i_array = canonical_population[alg_i]
    alg_j_array = canonical_population[alg_j]
    residual_terms_i = alg_i_array[residual_i_indices]
    residual_terms_j = alg_j_array[residual_j_indices]
    
    tensor_i = tensor_from_terms(residual_terms_i)
    tensor_j = tensor_from_terms(residual_terms_j)
    
    if np.array_equal(tensor_i, tensor_j):
        print(f'Pair ({alg_i}, {alg_j}): Tensors ARE equal')
    else:
        print(f'Pair ({alg_i}, {alg_j}): Tensors are NOT equal')


Pair (5832, 7414): Tensors ARE equal
Pair (9105, 10222): Tensors ARE equal
Pair (913, 1841): Tensors ARE equal
Pair (1618, 2156): Tensors ARE equal


### Results

The four highest-overlap pairs in the population are:

| Rule   | Algorithm pair | Shared terms |     Residual identity |
| ------ | -------------: | -----------: | --------------------: |
| Rule 0 |     5832, 7414 |           44 | $5\leftrightarrow5$ |
| Rule 1 |    9105, 10222 |           43 | $6\leftrightarrow6$ |
| Rule 2 |      913, 1841 |           42 | $7\leftrightarrow7$ |
| Rule 3 |     1618, 2156 |           42 | $7\leftrightarrow7$ |

Since each original decomposition contains 49 rank-one terms, removing the shared terms leaves two small residual decompositions for each pair.

For each rule, the tensors represented by the two residual decompositions were constructed exactly as

$$
R_i=\sum_{r\in I}u_r\otimes v_r\otimes w_r,
\qquad
R_j=\sum_{s\in J}\tilde u_s\otimes\tilde v_s\otimes\tilde w_s.
$$

Exact array comparison confirmed

$$
R_i=R_j
$$

for all four pairs. Thus, each high-overlap pair yields an exact local tensor identity in which one small collection of rank-one terms can be replaced by another without changing the tensor represented by the full algorithm.

We refer to these four identities as *Rules 0–3*.
